In [6]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import fetch_table_data, add_yoy_growth
import statsmodels.api as sm

In [19]:
start_date = '2015-01-01'
end_date = '2025-03-31'
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [23]:
trade_df = fetch_table_data(db_info, "korea_monthly_trade_data_forecast")

# pivot table 생성
pivot_df = trade_df.pivot(index='date', columns='root_hs_code', values='final_expDlr_yoy')
pivot_df.index = pd.to_datetime(pivot_df.index)
export_yoy_data = pivot_df.loc[start_date:end_date]

✅ 'korea_monthly_trade_data_forecast' 테이블에서 167220건의 데이터를 가져왔습니다.


In [32]:
# KSE_Price
price_df = fetch_table_data(db_info, "KSE_Price")

# 0) 준비: date를 datetime으로
price_df = price_df.copy()
price_df['date'] = pd.to_datetime(price_df['date'])
price_df = price_df.sort_values(['code', 'date'])

# 1) 월말로 기록된(=각 월의 마지막 거래일) 데이터 추출
#    (그 달의 가장 늦은 날짜 한 건만 선택)
period = price_df['date'].dt.to_period('M')
last_idx = price_df.groupby(['code', period])['date'].idxmax()
month_end_df = price_df.loc[last_idx, ['date', 'code', 'close']].copy()

# 2) date를 '진짜 달력 월말'로 재부여
month_end_df['date'] = month_end_df['date'].dt.to_period('M').dt.to_timestamp('M')

# 3) 월말 데이터 피벗 및 3/6/12개월 수익률 계산
monthly_close = (
    month_end_df
      .pivot(index='date', columns='code', values='close')
      .sort_index()
)

# (선택) 인덱스를 ‘모든 월말’로 보정해 결측을 명확히 드러냄
full_idx = pd.date_range(monthly_close.index.min(), monthly_close.index.max(), freq='M')
monthly_close = monthly_close.reindex(full_idx)

# 수익률: (현재/과거) - 1  —— 월별 시계열이므로 periods=3/6/12 사용
ret_3m  = monthly_close.pct_change(3)
ret_6m  = monthly_close.pct_change(6)
ret_12m = monthly_close.pct_change(12)

✅ 'KSE_Price' 테이블에서 6280830건의 데이터를 가져왔습니다.


In [37]:
ret_3m_rssize = ret_3m.loc[start_date:end_date].dropna(axis=1, how='any')
ret_6m_rssize = ret_6m.loc[start_date:end_date].dropna(axis=1, how='any')
ret_12m_rssize = ret_12m.loc[start_date:end_date].dropna(axis=1, how='any')

In [56]:
def calc_correlation_matrix(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    """
    두 개의 DataFrame을 입력받아 (df1.columns × df2.columns) 상관계수 매트릭스를 반환.
    - 공통 인덱스 기준으로 정렬 후 상관계수 계산
    - 피어슨 상관계수 사용
    """
    # 1. 공통 인덱스 찾기
    common_index = df1.index.intersection(df2.index)

    # 2. 공통 인덱스만 추출
    df1_common = df1.loc[common_index]
    df2_common = df2.loc[common_index]

    # 3. 상관계수 매트릭스 계산
    corr_df = pd.DataFrame(
        index=df1_common.columns,
        columns=df2_common.columns,
        dtype=float
    )

    for col1 in df1_common.columns:
        for col2 in df2_common.columns:
            corr_df.loc[col1, col2] = df1_common[col1].corr(df2_common[col2])

    return corr_df


def melt_pivot(df, id_name="root_hs_code", var_name="hs_code", value_name="value",
               dropna=True, sort=True):
    # 인덱스명이 없다면 안전하게 이름을 부여
    idx_name = df.index.name or id_name
    long_df = (
        df.reset_index()  # 인덱스를 컬럼으로
          .melt(id_vars=[idx_name], var_name=var_name, value_name=value_name)  # 축소
    )
    if dropna:
        long_df = long_df.dropna(subset=[value_name])
    if sort:
        long_df = long_df.sort_values([idx_name, var_name]).reset_index(drop=True)
    return long_df



In [42]:
correlation_3m_result = calc_correlation_matrix(ret_3m_rssize, export_yoy_data)
correlation_12m_result = calc_correlation_matrix(ret_12m_rssize, export_yoy_data)

In [47]:
correlation_6m_result = calc_correlation_matrix(ret_6m_rssize, export_yoy_data)

In [60]:
long_3m_df = melt_pivot(correlation_3m_result, value_name="correlation")
long_6m_df = melt_pivot(correlation_6m_result, value_name="correlation")
long_12m_df = melt_pivot(correlation_12m_result, value_name="correlation")

long_3m_df['period'] = '3m'
long_6m_df['period'] = '6m'
long_12m_df['period'] = '12m'

correlation_result = pd.concat([long_3m_df, long_6m_df, long_12m_df])

In [61]:
# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# DB에 저장
correlation_result.to_sql(
    name="correlation_price_hscode",  # 테이블명
    con=engine,
    if_exists='replace',  # 'append'로 하면 기존 데이터 뒤에 추가
    index=False           # DataFrame 인덱스는 저장 안 함
)

print("✅ 데이터가 'correlation_price_hscode' 테이블에 저장되었습니다.")

✅ 데이터가 'correlation_price_hscode' 테이블에 저장되었습니다.


In [55]:
correlation_3m_result

root_hs_code,121120,1212,121221,151550,151590,1518,170199,1902,190230,1905,...,903289,9301,9306,9401,940130,940199,940330,940540,950300,970191
code,,,,,,,,,,,,,,,,,,,,,
000020,-0.142576,-0.183932,-0.184513,0.293330,0.183652,0.011459,0.128800,0.153400,0.135832,0.085628,...,-0.124600,-0.046320,-0.000610,-0.301448,0.146917,0.117162,-0.167299,0.128266,-0.255487,0.242567
000040,-0.076821,0.012183,0.015233,-0.082253,0.016537,-0.007757,-0.005971,-0.000103,-0.022429,0.114429,...,0.186574,-0.011344,-0.038219,0.148649,-0.026730,0.317215,-0.031923,0.152443,-0.039624,0.195066
000050,0.002060,-0.022004,-0.021783,0.269442,0.473842,-0.006399,0.056375,-0.007431,-0.039316,0.089966,...,-0.128645,-0.036249,0.040886,-0.140282,-0.020450,-0.091805,-0.174143,0.051136,-0.075488,-0.032259
000070,0.046113,0.033637,0.028446,-0.117603,0.450104,-0.070187,0.053865,-0.064270,-0.067993,-0.020697,...,-0.096320,-0.072685,-0.026004,-0.093243,0.035073,-0.277017,-0.191542,0.067535,0.123118,0.589494
000080,-0.168225,-0.045399,-0.046571,0.184316,0.054956,-0.038260,-0.062904,-0.016016,-0.022440,-0.063575,...,-0.193362,0.063772,0.056937,-0.229097,-0.033822,0.214481,-0.039107,0.084132,-0.018078,-0.097489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900100,-0.113375,-0.117268,-0.114452,-0.033657,0.042462,-0.048725,-0.079421,-0.123561,-0.114856,-0.082007,...,-0.077828,-0.024760,-0.032781,-0.035926,0.176135,0.188737,0.101537,0.097831,-0.033115,0.121480
900110,-0.161520,-0.078472,-0.074614,0.092804,-0.006857,0.001738,-0.056287,-0.021044,-0.018353,0.024753,...,0.016990,0.019909,0.095708,-0.049425,-0.054787,-0.113441,-0.060588,-0.233592,-0.089958,-0.009891
900120,-0.110913,-0.139025,-0.137557,0.092901,0.159464,-0.058870,-0.039122,-0.033543,-0.061038,0.071008,...,0.019197,-0.045779,0.028902,-0.017258,-0.098926,0.414776,-0.044296,0.127109,-0.130829,-0.524724


In [ ]:
correlation_6m_result

In [ ]:
correlation_12m_result

In [58]:
long_3m_df = melt_pivot(correlation_3m_result, value_name="correlation")
long_6m_df = melt_pivot(correlation_6m_result, value_name="correlation")
long_12m_df = melt_pivot(correlation_12m_result, value_name="correlation")

long_3m_df['period'] = '3m'
long_6m_df['period'] = '6m'
long_12m_df['period'] = '12m'

correlaion_result = pd.concat([long_3m_df, long_6m_df, long_12m_df])

In [59]:
correlaion_result

,code,hs_code,correlation,period
0,000020,121120,-0.142576,3m
1,000020,1212,-0.183932,3m
2,000020,121221,-0.184513,3m
3,000020,151550,0.293330,3m
4,000020,151590,0.183652,3m
...,...,...,...,...
1178800,950130,940199,0.144811,12m
1178801,950130,940330,-0.211271,12m
1178802,950130,940540,0.145327,12m
1178803,950130,950300,0.072326,12m


In [75]:
correlation_12m_result.loc['011780'][['290124']]

root_hs_code
290124    0.18811
Name: 011780, dtype: float64

In [76]:
correlation_12m_result[['848690']]

root_hs_code,848690
code,
000020,-0.027965
000040,0.030107
000050,-0.033746
000070,0.002094
000080,0.083512
...,...
900100,-0.072291
900110,-0.066189
900120,0.152533
